# 📖 Notebook 1: Why Temporal?

Before we write any Temporal code, let's understand **why** workflow engines exist. We'll build a naive job processing system with Python and SQLite, then watch it fall apart.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why long-running, multi-step jobs are hard in distributed systems
- How the common "cron + database polling" approach fails
- What problems workflow engines like Temporal solve
- How Temporal's "durable execution" model works at a high level

## 🛠️ Setup

This notebook uses **only Python standard library** modules (no Docker needed).
We use SQLite (built into Python) to simulate a database.

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import sqlite3
import time
import random
import threading
from datetime import datetime

print("✅ All imports ready (standard library only — no installs needed)")

## 🤔 The Problem: Multi-Step Jobs

Imagine you're building an e-commerce system. When a customer places an order, you need to:

```
Step 1: Create the order in your database
         ↓
Step 2: Charge their credit card via Stripe
         ↓
Step 3: Reserve inventory in the warehouse
         ↓
Step 4: Schedule shipping with FedEx
         ↓
Step 5: Send a confirmation email
```

Each step calls a **different service**. Each **can fail**. Each takes a **different amount of time**.

**The critical question**: What happens when Step 3 fails **after** Step 2 already charged the customer's card?

Let's see how most teams try to solve this — and why it breaks.

## 🗄️ Attempt 1: The "Jobs Table" Approach

The most common first attempt is a database table of jobs + a cron/polling loop:

```
┌──────────────────────────────────────────────────┐
│  jobs table                                       │
│  id | type    | status  | payload | retry_count  │
│  1  | order   | pending | {...}   | 0            │
│  2  | order   | failed  | {...}   | 3            │
└──────────────────────────────────────────────────┘

Polling loop runs every few seconds:
  1. SELECT * FROM jobs WHERE status = 'pending'
  2. Process each job
  3. UPDATE jobs SET status = 'completed'
```

Let's build this and see what goes wrong.

In [ ]:
# Create an in-memory SQLite database to simulate our job system

db = sqlite3.connect(":memory:", check_same_thread=False)
db.execute("""
    CREATE TABLE jobs (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        type TEXT NOT NULL,
        status TEXT DEFAULT 'pending',
        payload TEXT,
        current_step INTEGER DEFAULT 0,
        retry_count INTEGER DEFAULT 0,
        error_message TEXT,
        created_at TEXT DEFAULT CURRENT_TIMESTAMP,
        updated_at TEXT DEFAULT CURRENT_TIMESTAMP
    )
""")

# Insert some sample order jobs
for i in range(1, 6):
    db.execute(
        "INSERT INTO jobs (type, payload) VALUES (?, ?)",
        ("order", f'{{"order_id": "ORD-{i:03d}", "amount": {random.randint(10, 500)}}}')
    )
db.commit()

# Show the jobs
print("📋 Our jobs table:")
print(f"{'ID':<4} {'Type':<8} {'Status':<10} {'Payload'}")
print("-" * 60)
for row in db.execute("SELECT id, type, status, payload FROM jobs"):
    print(f"{row[0]:<4} {row[1]:<8} {row[2]:<10} {row[3]}")

In [ ]:
# Simulate the multi-step order processing

def create_order(order_id):
    """Step 1: Create order in database."""
    time.sleep(0.1)  # simulate DB write
    return True

def charge_payment(order_id):
    """Step 2: Charge credit card (this calls Stripe)."""
    time.sleep(0.2)  # simulate API call
    return True

def reserve_inventory(order_id):
    """Step 3: Reserve items in warehouse — sometimes fails!"""
    time.sleep(0.1)
    if random.random() < 0.4:  # 40% failure rate
        raise Exception(f"Warehouse API timeout for {order_id}")
    return True

def ship_order(order_id):
    """Step 4: Schedule shipping."""
    time.sleep(0.1)
    return True

STEPS = [
    ("Create order",       create_order),
    ("Charge payment",     charge_payment),
    ("Reserve inventory",  reserve_inventory),
    ("Ship order",         ship_order),
]

print("✅ Defined 4 processing steps (Step 3 has a 40% failure rate)")

In [ ]:
# The naive job processor — polls the DB and processes pending jobs

def process_jobs():
    """Poll for pending jobs and process them one at a time."""
    rows = db.execute(
        "SELECT id, payload FROM jobs WHERE status = 'pending'"
    ).fetchall()

    if not rows:
        print("   No pending jobs.")
        return

    for job_id, payload in rows:
        import json
        data = json.loads(payload)
        order_id = data["order_id"]
        print(f"\n🔄 Processing job {job_id} (order {order_id})...")

        db.execute(
            "UPDATE jobs SET status = 'processing', updated_at = ? WHERE id = ?",
            (datetime.now().isoformat(), job_id)
        )
        db.commit()

        try:
            for step_num, (step_name, step_fn) in enumerate(STEPS, 1):
                print(f"   Step {step_num}: {step_name}...", end=" ")
                step_fn(order_id)
                print("✅")

            db.execute(
                "UPDATE jobs SET status = 'completed', current_step = ?, updated_at = ? WHERE id = ?",
                (len(STEPS), datetime.now().isoformat(), job_id)
            )
            db.commit()
            print(f"   ✅ Job {job_id} completed!")

        except Exception as e:
            print(f"❌ FAILED: {e}")
            db.execute(
                "UPDATE jobs SET status = 'failed', error_message = ?, updated_at = ? WHERE id = ?",
                (str(e), datetime.now().isoformat(), job_id)
            )
            db.commit()

# Run the processor
print("🏭 Running job processor...")
process_jobs()

In [ ]:
# Check results — some jobs will have failed

print("📊 Job Results:")
print(f"{'ID':<4} {'Status':<12} {'Error'}")
print("-" * 60)
for row in db.execute("SELECT id, status, error_message FROM jobs"):
    error = row[2] or "-"
    icon = "✅" if row[1] == "completed" else "❌" if row[1] == "failed" else "⏳"
    print(f"{row[0]:<4} {icon} {row[1]:<10} {error}")

# Count results
completed = db.execute("SELECT COUNT(*) FROM jobs WHERE status = 'completed'").fetchone()[0]
failed = db.execute("SELECT COUNT(*) FROM jobs WHERE status = 'failed'").fetchone()[0]
print(f"\n📈 Completed: {completed}, Failed: {failed}")

## 💥 Problem 1: Payment Was Charged, But Inventory Failed

Look at the failed jobs above. The payment (Step 2) **already succeeded** before the inventory
reservation (Step 3) failed. That means:

- ✅ The customer's credit card was **charged**
- ❌ But the order was **not fulfilled**
- 😡 The customer paid for nothing!

Our naive processor has **no way to undo** Step 2 when Step 3 fails.
We would need to write manual compensation logic — and that gets complicated fast.

## 💥 Problem 2: No Automatic Retries

When Step 3 fails due to a timeout, the job is just marked as `failed`.
But timeouts are **transient errors** — if we just tried again, it would probably work!

To add retries, we'd need to build:
- A retry counter
- Exponential backoff logic
- A way to distinguish retryable errors from permanent ones
- Logic to resume from the **correct step** (not restart from Step 1)

Let's try adding basic retries and see how messy it gets:

In [ ]:
# Adding retry logic to our processor — it gets messy quickly

MAX_RETRIES = 3

def process_jobs_with_retries():
    """A 'better' processor that retries failed jobs."""
    rows = db.execute(
        "SELECT id, payload, retry_count, current_step FROM jobs WHERE status = 'failed' AND retry_count < ?",
        (MAX_RETRIES,)
    ).fetchall()

    if not rows:
        print("   No retryable jobs.")
        return

    for job_id, payload, retry_count, current_step in rows:
        import json
        data = json.loads(payload)
        order_id = data["order_id"]

        # ⚠️ BUG: We restart from step 0, not from where we left off!
        # Fixing this requires tracking which steps completed — more complexity.
        print(f"\n🔁 Retrying job {job_id} (attempt {retry_count + 1}/{MAX_RETRIES})...")

        db.execute(
            "UPDATE jobs SET status = 'processing', retry_count = ?, updated_at = ? WHERE id = ?",
            (retry_count + 1, datetime.now().isoformat(), job_id)
        )
        db.commit()

        try:
            for step_num, (step_name, step_fn) in enumerate(STEPS, 1):
                print(f"   Step {step_num}: {step_name}...", end=" ")
                step_fn(order_id)
                print("✅")

            db.execute(
                "UPDATE jobs SET status = 'completed', updated_at = ? WHERE id = ?",
                (datetime.now().isoformat(), job_id)
            )
            db.commit()
            print(f"   ✅ Job {job_id} completed on retry!")

        except Exception as e:
            print(f"❌ FAILED again: {e}")
            db.execute(
                "UPDATE jobs SET status = 'failed', error_message = ?, updated_at = ? WHERE id = ?",
                (str(e), datetime.now().isoformat(), job_id)
            )
            db.commit()

print("🔁 Retrying failed jobs...")
process_jobs_with_retries()

print("\n⚠️  Notice the problem: we re-run ALL steps, including Step 2 (charge payment).")
print("   The customer could be charged TWICE! This is not idempotent.")

## 💥 Problem 3: No Visibility

When something goes wrong at 3 AM, you need to answer:
- Which step failed?
- What was the error message?
- How many times has it been retried?
- What was the input data?
- How long did each step take?

Our jobs table only stores `status = 'failed'` and an error message.
We have **no audit trail** of what happened. No timeline. No step-by-step history.

To get visibility, we'd need to build:
- An events table (one row per step per attempt)
- Timing information for each step
- A dashboard to visualize it all

That's a LOT of infrastructure code that has nothing to do with your business logic.

## 💥 Problem 4: Server Crashes

What if the server crashes **while processing** a job?

```
Step 1: Create order    ✅ (done)
Step 2: Charge payment  ✅ (done)
Step 3: Reserve invent— 💥 SERVER CRASHES
```

The job's status is still `'processing'` in the database. When the server restarts:
- Nobody picks up the job (it's not `'pending'` or `'failed'`)
- The customer was charged but the order is stuck
- You need a **separate cleanup process** to find stuck jobs

Let's simulate this:

In [ ]:
# Simulate a server crash during processing

db2 = sqlite3.connect(":memory:")
db2.execute("""
    CREATE TABLE jobs (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        status TEXT DEFAULT 'pending',
        payload TEXT
    )
""")
db2.execute("INSERT INTO jobs (payload) VALUES ('order-crash-test')")
db2.commit()

# Start processing...
print("🔄 Processing job 1...")
db2.execute("UPDATE jobs SET status = 'processing' WHERE id = 1")
db2.commit()
print("   Step 1: Create order ✅")
print("   Step 2: Charge payment ✅")
print("   Step 3: Reserve invent— 💥 SERVER CRASHES!")
print()

# Server restarts — check what the new process sees
print("🔄 Server restarts. Checking for pending jobs...")
pending = db2.execute("SELECT COUNT(*) FROM jobs WHERE status = 'pending'").fetchone()[0]
processing = db2.execute("SELECT COUNT(*) FROM jobs WHERE status = 'processing'").fetchone()[0]
print(f"   Pending jobs: {pending}")
print(f"   Stuck 'processing' jobs: {processing}")
print()
print("😱 The job is STUCK! It's not 'pending' so the processor won't pick it up.")
print("   The customer was charged but the order is incomplete.")
print("   You need MANUAL intervention or a separate cleanup process.")

## 📋 Summary of Problems with Cron + DB Polling

| Problem | Description | Impact |
|---------|-------------|--------|
| **No compensation** | Can't undo completed steps when later steps fail | Customer charged but order not fulfilled |
| **No smart retries** | Basic retry re-runs ALL steps, not just the failed one | Double charges, duplicate work |
| **No visibility** | Only know "failed" — not which step, when, or why | Debugging at 3 AM is a nightmare |
| **Crash recovery** | Jobs get stuck in `processing` state forever | Requires manual cleanup |
| **No rate limiting** | Burst of jobs can overwhelm downstream services | Cascade failures |
| **Polling waste** | Checking every N seconds burns CPU even when idle | Wasted resources |

Every team that builds their own job system eventually reinvents these solutions.
**Workflow engines exist so you don't have to.**

## 🚀 Enter Temporal: Durable Execution

**Temporal** is an open-source workflow orchestration engine. It solves ALL the problems above.

### The Key Idea: Your Code is the Workflow

With Temporal, you write your business logic as normal Python code:

```python
@workflow.defn
class OrderWorkflow:
    @workflow.run
    async def run(self, order_id: str):
        await workflow.execute_activity(create_order, order_id, ...)
        await workflow.execute_activity(charge_payment, order_id, ...)
        await workflow.execute_activity(reserve_inventory, order_id, ...)
        await workflow.execute_activity(ship_order, order_id, ...)
```

This looks like normal code. But Temporal makes it **indestructible**:

| Problem | How Temporal Solves It |
|---------|----------------------|
| Crash recovery | Replays workflow from event history — resumes exactly where it left off |
| Retries | Built-in retry policies with exponential backoff |
| Compensation | Saga pattern support — run undo steps in reverse on failure |
| Visibility | Full event history for every workflow — see each step, timing, errors |
| Rate limiting | Built-in task queue throttling |
| Polling waste | Workers use long-polling — no wasted CPU |

## 🏗️ How Temporal Works (Architecture)

Temporal has three main pieces:

```
┌─────────────┐     ┌──────────────────┐     ┌───────────────────┐
│   Client     │────▶│  Temporal Server  │────▶│     Worker        │
│  (your app)  │     │  (the brain)      │     │  (runs your code) │
│              │     │                   │     │                   │
│ "Start this  │     │ "I'll track every │     │ "I'll execute     │
│  workflow"   │     │  step as events"  │     │  the activities"  │
└─────────────┘     └────────┬──────────┘     └───────────────────┘
                             │
                    ┌────────▼──────────┐
                    │    PostgreSQL      │
                    │  (event store)     │
                    └───────────────────┘
```

### The Three Roles

1. **Client** — Your application code that says "please run OrderWorkflow for order-123"
2. **Temporal Server** — Records every step as an event in the database. Schedules tasks for workers.
3. **Worker** — A Python process that polls Temporal for tasks and executes your workflow/activity code

### The Magic: Event Sourcing

Every time something happens in your workflow, Temporal writes an **event**:

```
Event 1: WorkflowExecutionStarted  (order_id = "ORD-001")
Event 2: ActivityTaskScheduled      (create_order)
Event 3: ActivityTaskCompleted      (create_order → success)
Event 4: ActivityTaskScheduled      (charge_payment)
Event 5: ActivityTaskCompleted      (charge_payment → success)
Event 6: ActivityTaskScheduled      (reserve_inventory)
Event 7: ActivityTaskFailed         (reserve_inventory → timeout)
Event 8: ActivityTaskScheduled      (reserve_inventory)  ← automatic retry!
Event 9: ActivityTaskCompleted      (reserve_inventory → success)
...
```

If the server crashes after Event 5, Temporal **replays** events 1-5 to reconstruct the workflow's state, then continues from Event 6. The payment is NOT re-charged because Temporal knows it already completed.

## 🆚 Comparison: Before and After Temporal

Let's compare the same order processing scenario:

In [ ]:
# Side-by-side comparison: what you have to build yourself vs what Temporal gives you

comparison = [
    ("Retry logic",
     "Build retry counter, backoff, error classification",
     "RetryPolicy(max_attempts=5, backoff=2.0)"),
    ("Crash recovery",
     "Cleanup cron, stuck job detector, state machine",
     "Automatic — replays from event history"),
    ("Compensation",
     "Manual rollback code, reverse-order execution",
     "Saga pattern with try/except"),
    ("Visibility",
     "Build events table, dashboard, alerting",
     "Built-in Web UI with full event history"),
    ("Rate limiting",
     "Token bucket, sliding window, custom code",
     "Task queue configuration"),
    ("Long waits",
     "Cron-scheduled wake-ups, state in DB",
     "workflow.sleep(timedelta(days=30))"),
    ("Human approval",
     "Polling endpoint, webhook handler, state machine",
     "workflow.wait_condition() + signals"),
]

print("🆚 Build-It-Yourself vs Temporal")
print("=" * 90)
print(f"{'Feature':<18} {'DIY (Cron + DB)':<40} {'Temporal'}")
print("-" * 90)
for feature, diy, temporal in comparison:
    print(f"{feature:<18} {diy:<40} {temporal}")

print()
print("💡 Temporal lets you focus on your BUSINESS LOGIC instead of infrastructure.")

## 🏢 Who Uses Temporal?

Temporal (and its predecessor, Cadence at Uber) is used by major tech companies:

| Company | Use Case |
|---------|----------|
| **Uber** | Trip lifecycle, payment processing, driver onboarding |
| **Netflix** | Media encoding pipelines, content delivery |
| **Snap** | Ad delivery, content moderation |
| **Stripe** | Payment orchestration across multiple providers |
| **Coinbase** | Cryptocurrency transaction processing |
| **Datadog** | Infrastructure provisioning |

Microsoft built a similar concept with **Azure Durable Functions**.
AWS has **Step Functions**. Google Cloud has **Workflows**.

The idea of durable execution is becoming an industry standard.

## 🧠 Key Vocabulary

Before we start coding with Temporal, make sure you know these terms:

| Term | Plain English | Analogy |
|------|--------------|----------|
| **Workflow** | A sequence of steps that must complete reliably | A recipe |
| **Activity** | A single step that does real work (API call, DB write) | One cooking step |
| **Worker** | A process that executes workflow/activity code | A cook in the kitchen |
| **Task Queue** | A named channel that connects clients to workers | The order ticket rail |
| **Event History** | A log of everything that happened in a workflow | The recipe's execution log |
| **Signal** | A message sent to a running workflow from outside | Telling the cook "add extra cheese" |
| **Timer** | A workflow pause that resumes after a duration | "Let the dough rise for 1 hour" |
| **Saga** | A pattern for undoing completed steps on failure | Returning ingredients if the recipe fails |

## 📚 Summary

### What We Learned

1. **Multi-step jobs are hard** — each step can fail, and you need to handle partial completion
2. **Cron + DB polling breaks** — no retries, no compensation, no crash recovery, no visibility
3. **Building these features yourself is a full-time job** — retry logic, state machines, event tables, dashboards
4. **Temporal provides durable execution** — your code looks normal, but it survives crashes and retries automatically
5. **Event sourcing is the key** — Temporal records every step, enabling replay and recovery

### Next Up

In **Notebook 2**, we'll write our first Temporal workflow and activity, connect to a local Temporal server,
and see durable execution in action!